# Judge validation v2
Prepare disjoint ID-only subsets, compare two prompts on dev, then evaluate one frozen candidate on untouched validation.

In [ ]:
from pathlib import Path

from rubric_order.api import (
    compact_judge_plan,
    estimate_judge_validation,
    find_project_root,
    installed_model_provenance,
    load_experiment_config,
    prepare_judge_validation,
    run_judge_dev,
    run_judge_holdout,
)

PROJECT_ROOT = find_project_root(Path.cwd())

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "judge_validation.yaml"
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
RUN_ID = "judge-validation-v2"
RUNTIME_OVERRIDES = {}
CHECK_MODEL_PROVENANCE = False
PREPARE_SUBSETS = False
RUN_DRY_RUN = False
EXECUTE_MODEL_CALLS = False
MAX_CALLS = 0
DEV_CANDIDATE = "judge_absolute"
MODEL_SELECTION = "llama"
RUN_UNTOUCHED_VALIDATION = False

In [ ]:
config = load_experiment_config(CONFIG_PATH, DATA_PATH, overrides=RUNTIME_OVERRIDES)
print({"run_id": RUN_ID, "execute": EXECUTE_MODEL_CALLS, "max_calls": MAX_CALLS})
estimate_judge_validation(config)

In [ ]:
if CHECK_MODEL_PROVENANCE:
    display(installed_model_provenance(config))
else:
    print("Read-only model provenance check skipped.")

In [ ]:
if PREPARE_SUBSETS:
    manifest = prepare_judge_validation(config, RUN_ID, resume=False)
    display(manifest)
else:
    print("Subset preparation skipped; no run directory was created or changed.")

In [ ]:
if RUN_DRY_RUN:
    dry_result = run_judge_dev(
        config,
        RUN_ID,
        execute_model_calls=False,
        max_calls=0,
        model=MODEL_SELECTION,
    )
    display(compact_judge_plan(dry_result))
else:
    print("Dry-run skipped; set RUN_DRY_RUN=True to write a sanitized external manifest.")

In [ ]:
if EXECUTE_MODEL_CALLS:
    dev_result = run_judge_dev(
        config,
        RUN_ID,
        execute_model_calls=True,
        max_calls=MAX_CALLS,
        candidate=DEV_CANDIDATE,
        model=MODEL_SELECTION,
    )
    display(dev_result)
else:
    print(
        "Judge-dev inference disabled. Restore EXECUTE_MODEL_CALLS=False and "
        "MAX_CALLS=0 after each batch."
    )

In [ ]:
if RUN_UNTOUCHED_VALIDATION and EXECUTE_MODEL_CALLS:
    display(run_judge_holdout(config, RUN_ID, execute_model_calls=True, max_calls=MAX_CALLS))
else:
    print(
        "Untouched validation remains disabled until judge-dev has a frozen gate-passing candidate."
    )